# 🧬 TIDEST quick-start

**TIDEST** (Testing Imputed Differential Expressions for Spatial Transcriptomics) finds genes that differ between two spatial groups *after* a deep-learning imputation step — while correcting the imputation's systematic errors and adjusting for latent spatial structure that would otherwise create false positives.

This notebook runs the whole pipeline on a small **synthetic** dataset in a few seconds (no downloads, no R required) and shows, with pictures:

1. what the synthetic tissue looks like,
2. how TIDEST's *augmented outcome* cleans up the noisy imputation,
3. which genes it recovers, and
4. how it compares to a naïve t-test under spatial confounding.

Run from the repository root so `examples/_synthetic.py` is importable.

In [ ]:
import sys, os
# Put examples/ on the path so `_synthetic` is importable, and drop the working
# directory so the local ./tidest source folder cannot shadow the *installed*
# tidest package (run `pip install -e tidest` / `make install` first).
_root = os.getcwd()
_examples = os.path.join(_root, 'examples')
_examples = _examples if os.path.isdir(_examples) else _root
sys.path = [p for p in sys.path if os.path.abspath(p or '.') != _root]
sys.path.insert(0, _examples)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import roc_auc_score

from tidest import tidest
from _synthetic import make_dataset

# A clean, consistent look
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.25})
TIDEST_C, TTEST_C, RAW_C = '#C44E52', '#4C72B0', '#9AA0A6'

data = make_dataset(N=400, G_DE=30, G_null=70, M=10, n_pcs=20, seed=0)
genes  = data['genes']
is_de  = data['is_de']
coords = data['coords']
print(f"{data['st_adata'].n_obs} spots × {len(genes)} genes  —  "
      f"{int(is_de.sum())} truly DE, {int((~is_de).sum())} null")

## 1. Peek at the synthetic tissue

Each spot has a binary **treatment** `A` (think: two tissue regions), an unobserved **spatial confounder** `Z` (a smooth field — e.g. cell-density or anatomy), and gene expression driven by *both*. Because `Z` lines up with the region boundary, a naïve comparison will blame `Z`'s effect on the treatment.

In [ ]:
de_gene = int(np.argmax(np.abs(data['tau_true'])))   # the strongest true DE gene
panels = [('Treatment A', data['A'], 'coolwarm'),
          ('Latent confounder Z', data['Z'], 'viridis'),
          (f'Expression — gene{de_gene} (true DE)', data['Y_true'][:, de_gene], 'magma')]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (title, c, cmap) in zip(axes, panels):
    sc = ax.scatter(coords[:, 0], coords[:, 1], c=c, cmap=cmap, s=22, edgecolor='none')
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_aspect('equal'); fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle('Synthetic spatial tissue', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 2. Fit TIDEST

We pass the single-cell reference (for the gene–gene correlations), the spatial counts, the imputed expression, and a precomputed spatial-PC matrix `U` (so SpatialPCA/R is skipped). The treatment is handed in as an array.

In [ ]:
model = tidest(n_pcs=20, n_folds=2, corr_threshold=0.5, seed=0)
model.fit(
    sc_adata=data['sc_adata'],
    st_adata=data['st_adata'],
    pred_adata=data['pred_adata'],
    U=data['U'],
    treatment=data['A'],
    genes=genes,
    pred_is_log=True,
)
res = model.results_.set_index('gene')
res.head()

## 3. The augmented outcome cleans up the imputation

TIDEST corrects each imputed gene using the observed prediction errors of its most-correlated neighbours. Compared against the *true* latent expression, the augmented outcome has a much lower reconstruction error than the raw imputation (this is what decouples downstream testing from imputer quality).

In [ ]:
Y_true = data['Y_true']
Y_raw  = np.asarray(data['pred_adata'][:, genes].X)
Y_aug  = np.asarray(model.pseudo_[data['st_adata'].obs_names, genes].X)
rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
rmse_raw, rmse_aug = rmse(Y_raw, Y_true), rmse(Y_aug, Y_true)
drop = 100 * (1 - rmse_aug / rmse_raw)

fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(13, 4))
lim = [min(Y_true.min(), Y_raw.min()), max(Y_true.max(), Y_raw.max())]
for ax, Y, title, col in [(ax0, Y_raw, f'Raw imputation\nRMSE = {rmse_raw:.2f}', RAW_C),
                          (ax1, Y_aug, f'Augmented (TIDEST)\nRMSE = {rmse_aug:.2f}', TIDEST_C)]:
    ax.scatter(Y_true.ravel(), Y.ravel(), s=4, alpha=0.15, color=col, edgecolor='none')
    ax.plot(lim, lim, 'k--', lw=1)
    ax.set_xlabel('true expression'); ax.set_ylabel('reconstructed'); ax.set_title(title)
    ax.set_aspect('equal')
ax2.bar(['Raw\nimputation', 'Augmented\n(TIDEST)'], [rmse_raw, rmse_aug], color=[RAW_C, TIDEST_C])
ax2.set_ylabel('reconstruction RMSE'); ax2.set_title(f'↓ {drop:.0f}% error')
ax2.grid(axis='x')
plt.tight_layout(); plt.show()
print(f'Augmentation reduces reconstruction RMSE by {drop:.0f}%.')

## 4. Did we recover the right genes?

Left: estimated effect $\hat\tau$ vs the ground truth (points on the dashed line are perfectly estimated). Right: a volcano plot — true DE genes (red) should sit high and away from zero; nulls (grey) should stay near the centre and below the significance line.

In [ ]:
r = res.loc[genes]
tau_hat, tau_true = r['tau'].values, data['tau_true']
qval = r['qval'].values
sig  = qval < 0.05
colors = np.where(is_de, TIDEST_C, RAW_C)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.6))
lim = [min(tau_true.min(), tau_hat.min()) - 0.1, max(tau_true.max(), tau_hat.max()) + 0.1]
ax0.plot(lim, lim, 'k--', lw=1, zorder=0)
ax0.scatter(tau_true, tau_hat, c=colors, s=40, edgecolor='white', linewidth=0.5)
ax0.set_xlabel('true effect  τ'); ax0.set_ylabel('estimated effect  τ̂')
ax0.set_title('Effect recovery'); ax0.set_aspect('equal')

neglogq = -np.log10(np.clip(qval, 1e-300, 1))
ax1.scatter(tau_hat, neglogq, c=colors, s=40, edgecolor='white', linewidth=0.5)
ax1.axhline(-np.log10(0.05), color='k', ls='--', lw=1)
ax1.text(lim[0], -np.log10(0.05) + 0.3, 'q = 0.05', fontsize=9)
ax1.set_xlabel('estimated effect  τ̂'); ax1.set_ylabel('−log₁₀ q')
ax1.set_title('Volcano')
handles = [plt.Line2D([], [], marker='o', ls='', color=TIDEST_C, label='true DE'),
           plt.Line2D([], [], marker='o', ls='', color=RAW_C, label='null')]
ax1.legend(handles=handles, loc='upper right', frameon=False)
plt.tight_layout(); plt.show()

## 5. TIDEST vs a naïve t-test

We run a Welch t-test per gene on the observed counts and compare the two methods on **power** (true positives caught), **false-positive rate** (nulls wrongly called), and **AUC** (ranking quality). Under spatial confounding the t-test inflates its false positives, while TIDEST keeps them controlled at comparable power.

In [ ]:
# Naïve Welch t-test on the observed (log1p) counts, per gene
X = np.asarray(data['st_adata'][:, genes].X)
A = data['A'].astype(bool)
t_stat, p_t = stats.ttest_ind(X[A], X[~A], axis=0, equal_var=False)
q_t = multipletests(np.nan_to_num(p_t, nan=1.0), method='fdr_bh')[1]

def perf(qv, score):
    s = qv < 0.05
    tpr = (s & is_de).sum() / is_de.sum()
    fpr = (s & ~is_de).sum() / (~is_de).sum()
    auc = roc_auc_score(is_de.astype(int), score)
    return tpr, fpr, auc

tpr_T, fpr_T, auc_T = perf(qval, np.abs(r['z'].values))
tpr_t, fpr_t, auc_t = perf(q_t,  np.abs(t_stat))

metrics = ['Power (TPR)', 'False-positive rate', 'AUC']
tide_v  = [tpr_T, fpr_T, auc_T]
ttest_v = [tpr_t, fpr_t, auc_t]
x = np.arange(len(metrics)); w = 0.36
fig, ax = plt.subplots(figsize=(8, 4.4))
b1 = ax.bar(x - w/2, tide_v,  w, label='TIDEST', color=TIDEST_C)
b2 = ax.bar(x + w/2, ttest_v, w, label='t-test', color=TTEST_C)
ax.axhline(0.05, color='k', ls=':', lw=1)
ax.text(1.0, 0.07, 'nominal 5%', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(metrics); ax.set_ylim(0, 1.05)
ax.set_title('TIDEST vs naïve t-test'); ax.legend(frameon=False); ax.grid(axis='x')
for bars in (b1, b2):
    for bar in bars:
        ax.annotate(f'{bar.get_height():.2f}', (bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Sign accuracy on TIDEST's detected DE genes
det = sig & is_de
sign_acc = np.mean(np.sign(tau_hat[det]) == np.sign(tau_true[det])) if det.any() else float('nan')

print('TIDEST summary')
print(f'  detected {int(sig.sum())}/{len(genes)} genes at q < 0.05')
print(f'  power = {tpr_T:.2f}   FPR = {fpr_T:.2f}   AUC = {auc_T:.3f}   sign-accuracy = {sign_acc:.2f}')
print('naïve t-test')
print(f'  power = {tpr_t:.2f}   FPR = {fpr_t:.2f}   AUC = {auc_t:.3f}')
print(f'\n→ TIDEST cuts the false-positive rate by {fpr_t/max(fpr_T,1e-9):.1f}× at comparable power.')

## Recap

On this synthetic tissue TIDEST (1) **denoises** the imputation, cutting reconstruction error sharply, (2) **recovers** the true DE genes with the correct sign, and (3) **controls false positives** under spatial confounding far better than a naïve test — the same behaviour demonstrated at scale in the TIDEST paper.

Swap in your own `sc_adata`, `st_adata`, and imputed `pred_adata` (and drop the `U=` argument to let TIDEST estimate spatial confounders with SpatialPCA) to run it on real data. See [`../REPRODUCE.md`](../REPRODUCE.md) for the full analyses.